### Install dependencies

This notebook requires a dependency which can be installed with the following command `pip install open-bus-stride-client`.

You can also launch it online at [this URL](https://mybinder.org/v2/gh/hasadna/open-bus-stride-client/main?labpath=notebooks%2Fload%20siri%20vehicle%20locations%20to%20pandas%20dataframe.ipynb), when launching online the dependencies are already installed.


In [1]:
# !pip install open-bus-stride-client

In [2]:
import pandas as pd
import datetime
from dateutil import tz

pd.options.display.max_columns = 1000
pd.options.display.max_colwidth = 1000

import stride

### Find a route to investigave

As SIRI data doesn't hold the `route_short_name` data (the bus line number) we will use the GTFS to find a route.

Let's look for line number `480` (Egged).

In [3]:
pd.DataFrame(stride.get('/gtfs_routes/list', {'route_short_name':5,
                                                            'agency_name': 'דן',
                                                            # 'route_long_name_contains': 'ירושלים',
                                              'date_from': '2026-07-29',
                                              'date_to':  '2026-07-30'}
                        ))

,id,date,line_ref,operator_ref,route_short_name,route_long_name,route_mkt,route_direction,route_alternative,agency_name,route_type
0,9334460,2026-07-29,2259,5,5,ת. רכבת תל אביב סבידור/על פרשת דרכים-תל אביב יפו<->מסוף הלוחמים/משרד הרישוי-תל אביב יפו-10,25005,1,0,דן,3
1,9334461,2026-07-29,2261,5,5,משרד הרישוי/הלוחמים-חולון<->ת. רכבת תל אביב - סבידור/הורדה-תל אביב יפו-20,25005,2,0,דן,3
2,9334462,2026-07-29,2262,5,5,אזור התעשייה-בני ברק<->בית העלמין/חזון אי''ש-בני ברק-10,26005,1,0,דן,3
3,9334463,2026-07-29,2263,5,5,בית העלמין/חזון אי''ש-בני ברק<->אזור התעשייה-בני ברק-20,26005,2,0,דן,3
4,9338796,2026-07-29,32055,5,5,אזור התעשייה-בני ברק<->אזור התעשייה-בני ברק-11,26005,1,1,דן,3
5,9340293,2026-07-30,2259,5,5,ת. רכבת תל אביב סבידור/על פרשת דרכים-תל אביב יפו<->מסוף הלוחמים/משרד הרישוי-תל אביב יפו-10,25005,1,0,דן,3
6,9340294,2026-07-30,2261,5,5,משרד הרישוי/הלוחמים-חולון<->ת. רכבת תל אביב - סבידור/הורדה-תל אביב יפו-20,25005,2,0,דן,3
7,9340295,2026-07-30,2262,5,5,אזור התעשייה-בני ברק<->בית העלמין/חזון אי''ש-בני ברק-10,26005,1,0,דן,3
8,9340296,2026-07-30,2263,5,5,בית העלמין/חזון אי''ש-בני ברק<->אזור התעשייה-בני ברק-20,26005,2,0,דן,3
9,9344642,2026-07-30,32055,5,5,אזור התעשייה-בני ברק<->אזור התעשייה-בני ברק-11,26005,1,1,דן,3


This line as 4 `line_ref` (routes), let's pick `7020` route and get its SIRI data.

### Get rides data

We use the stride iterate method to efficiently iterate over a possibly long list of results.

Behind the scenes it uses the offset/limit parameters so you don't have to worry about it.

We pass on the iterator directly on to Pandas to create a DataFrame.

In [4]:
siri_vehicle_locations = pd.DataFrame(stride.iterate('/siri_vehicle_locations/list', {
    'siri_routes__line_ref': '32055',
    'siri_rides__schedualed_start_time_from': datetime.datetime(2026,7, 29, tzinfo=tz.gettz('Israel')),
    'siri_rides__schedualed_start_time_to': datetime.datetime(2026,7, 30, tzinfo=tz.gettz('Israel'))+datetime.timedelta(days=1),
    'order_by': 'recorded_at_time desc'
}, limit=1000000))

siri_vehicle_locations.shape

(100, 22)

In [5]:
siri_vehicle_locations[['recorded_at_time','siri_route__line_ref',
                                    'siri_route__operator_ref','siri_ride__scheduled_start_time',
                                   'lon','lat','siri_ride__vehicle_ref']].head(20)

,recorded_at_time,siri_route__line_ref,siri_route__operator_ref,siri_ride__scheduled_start_time,lon,lat,siri_ride__vehicle_ref
0,2026-07-30 09:06:05+00:00,32055,5,2026-07-30 08:35:00+00:00,34.833817,32.087391,53104403
1,2026-07-30 09:05:00+00:00,32055,5,2026-07-30 09:05:00+00:00,34.828529,32.102116,17774403
2,2026-07-30 09:05:00+00:00,32055,5,2026-07-30 09:05:00+00:00,34.828529,32.102116,17774403
3,2026-07-30 09:05:00+00:00,32055,5,2026-07-30 08:35:00+00:00,34.833828,32.089733,53104403
4,2026-07-30 09:04:24+00:00,32055,5,2026-07-30 07:30:00+00:00,34.826687,32.098431,53107003
5,2026-07-30 09:04:23+00:00,32055,5,2026-07-30 08:35:00+00:00,34.833675,32.090572,53104403
6,2026-07-30 09:04:15+00:00,32055,5,2026-07-30 09:05:00+00:00,34.829144,32.101685,17774403
7,2026-07-30 09:03:09+00:00,32055,5,2026-07-30 08:35:00+00:00,34.833260,32.092236,53104403
8,2026-07-30 09:03:01+00:00,32055,5,2026-07-30 07:30:00+00:00,34.825775,32.098190,53107003
9,2026-07-30 09:03:01+00:00,32055,5,2026-07-30 07:30:00+00:00,34.825775,32.098190,53107003


The date columns are on UTC timezone, let's localize the dates to Israel timezone.

In [6]:
def localize_dates(data, dt_columns = None):
    if dt_columns is None:
        dt_columns=[]
    
    data = data.copy()
    
    for c in dt_columns:
        data[c] = pd.to_datetime(data[c]).dt.tz_convert('Israel')
    
    return data

In [7]:
dt_columns = ['recorded_at_time','siri_ride__scheduled_start_time']

siri_vehicle_locations = localize_dates(siri_vehicle_locations, dt_columns)

In [8]:
siri_vehicle_locations.siri_ride__scheduled_start_time.value_counts().sort_index()

siri_ride__scheduled_start_time
2026-07-30 09:15:00+03:00    19
2026-07-30 09:45:00+03:00     4
2026-07-30 10:15:00+03:00     8
2026-07-30 10:30:00+03:00    18
2026-07-30 10:50:00+03:00    19
2026-07-30 11:20:00+03:00    11
2026-07-30 11:35:00+03:00    10
2026-07-30 11:50:00+03:00     8
2026-07-30 12:05:00+03:00     3
Name: count, dtype: int64

It looks great! (*note 18/03/2022 is Friday*)

Now we can use Pandas to get some information from this data.

### Notes and Resources

siri_rides/list: 
- `siri_route_ids`: route_ids field can be a comma-separated string containing a list of ids.
- all date/time parameters must have a timezone (for example: `datetime.datetime.now(datetime.timezone.utc) - datetime.timedelta(days=1)`).
- `order_by`: any field can be specified in `order_by` with asc or desc specifier, you can specify comma-separated multiple values.
- `limit`: any number can be specified for the limit as we use pagination behind the scenes, default is 10,000.

Documentation: https://open-bus-stride-api.hasadna.org.il/docs#/

In [9]:
import folium
import branca.colormap as cm

# SIRI data has many pings across different vehicles/rides mixed together - pick the
# (vehicle, scheduled start time) combo with the most recorded pings so the path traces
# one continuous, well-sampled trip instead of jumbling several vehicles together
ride_key = (
    siri_vehicle_locations.groupby(['siri_ride__vehicle_ref', 'siri_ride__scheduled_start_time'])
    .size().idxmax()
)
vehicle_ref, scheduled_start_time = ride_key
ride = siri_vehicle_locations[
    (siri_vehicle_locations['siri_ride__vehicle_ref'] == vehicle_ref) &
    (siri_vehicle_locations['siri_ride__scheduled_start_time'] == scheduled_start_time)
].sort_values('recorded_at_time')

start_time = ride['recorded_at_time'].iloc[0]
elapsed_min = (ride['recorded_at_time'] - start_time).dt.total_seconds() / 60

# explicit width/height (not the default responsive iframe) - some notebook front-ends
# (e.g. VS Code) collapse folium's default 0-height responsive container to nothing
vehicle_map = folium.Map(location=[ride['lat'].mean(), ride['lon'].mean()], zoom_start=14, tiles='OpenStreetMap',
                          width=900, height=600)

colormap = cm.LinearColormap(
    colors=['#440154', '#31688e', '#35b779', '#fde725'],  # viridis-style gradient
    vmin=elapsed_min.min(), vmax=elapsed_min.max()
)
colormap.caption = f"Minutes since {start_time.strftime('%H:%M')} (vehicle {vehicle_ref})"

coords = list(zip(ride['lat'], ride['lon']))

for (lat1, lon1), (lat2, lon2), t in zip(coords[:-1], coords[1:], elapsed_min.iloc[:-1]):
    folium.PolyLine([(lat1, lon1), (lat2, lon2)], color=colormap(t), weight=5, opacity=0.9).add_to(vehicle_map)

for (lat, lon), t, rec in zip(coords, elapsed_min, ride['recorded_at_time']):
    folium.CircleMarker(
        location=(lat, lon),
        radius=5,
        color='black',
        weight=1,
        fill=True,
        fill_color=colormap(t),
        fill_opacity=1.0,
        popup=f"recorded at {rec.strftime('%H:%M:%S')}",
    ).add_to(vehicle_map)

colormap.add_to(vehicle_map)
vehicle_map